# Gemini Live API PoC — auto VAD + inputTranscription 타이밍 검증

**Phase 0 목표**: `inputTranscription`이 `modelTurn`보다 먼저(또는 동시에) 도착하는지 확인.
- **YES** → parallel pipeline 채택 (audio forward + classifier 병렬)
- **NO** → hold-and-scan 또는 push-to-talk fallback

**모델**: `gemini-3.1-flash-live-preview` (AUDIO 전용, TEXT modality 미지원)

In [ ]:
import asyncio
import io
import logging
import os
import subprocess
import tempfile
import time

import av
from google import genai
from google.genai import types

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s.%(msecs)03d %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("poc")

MODEL = "models/gemini-3.1-flash-live-preview"
RECV_TIMEOUT = 15

## Audio helpers

macOS `say` 로 TTS 생성 → `av` 로 PCM s16le 16kHz mono 변환

In [ ]:
def tts_to_pcm(text: str, lang: str = "en") -> bytes:
    """Use macOS `say` to synthesize speech, convert to PCM s16le 16kHz mono."""
    voice = "Yuna" if lang == "ko" else "Samantha"
    with tempfile.NamedTemporaryFile(suffix=".aiff", delete=True) as tmp:
        subprocess.run(
            ["say", "-v", voice, "-o", tmp.name, text],
            check=True, capture_output=True,
        )
        return aiff_to_pcm(tmp.name)


def aiff_to_pcm(path: str, target_rate: int = 16000) -> bytes:
    """Convert audio file to PCM s16le mono at target_rate using av."""
    output = io.BytesIO()
    container = av.open(path)
    resampler = av.AudioResampler(format="s16", layout="mono", rate=target_rate)
    for frame in container.decode(audio=0):
        for resampled in resampler.resample(frame):
            output.write(resampled.planes[0])
    container.close()
    return output.getvalue()

## Event collector + audio turn runner

In [ ]:
async def collect_events(session, label: str, t0: float, timeout: float = RECV_TIMEOUT) -> list[dict]:
    """Receive server events until turnComplete/interrupted or timeout."""
    events = []

    async def _recv():
        async for msg in session.receive():
            elapsed_ms = (time.monotonic() - t0) * 1000
            sc = msg.server_content
            if sc is None:
                continue
            if sc.input_transcription:
                events.append({
                    "type": "inputTranscription",
                    "elapsed_ms": round(elapsed_ms, 1),
                    "text": sc.input_transcription.text or "",
                })
            if sc.model_turn:
                parts = sc.model_turn.parts or []
                text_parts = [p.text for p in parts if p.text]
                has_audio = any(p.inline_data for p in parts if hasattr(p, "inline_data") and p.inline_data)
                desc = "".join(text_parts)[:80] if text_parts else ("(audio)" if has_audio else "(empty)")
                events.append({"type": "modelTurn", "elapsed_ms": round(elapsed_ms, 1), "text": desc})
            if sc.turn_complete:
                events.append({"type": "turnComplete", "elapsed_ms": round(elapsed_ms, 1)})
                return
            if sc.interrupted:
                events.append({"type": "interrupted", "elapsed_ms": round(elapsed_ms, 1)})
                return

    try:
        await asyncio.wait_for(_recv(), timeout=timeout)
    except asyncio.TimeoutError:
        events.append({"type": "TIMEOUT", "elapsed_ms": round((time.monotonic() - t0) * 1000, 1)})
    return events


async def run_audio_turn(session, text: str, lang: str, label: str) -> dict:
    """Synthesize speech, send as audio realtimeInput, collect timing."""
    pcm = tts_to_pcm(text, lang=lang)
    print(f"[{label}] PCM: {len(pcm)} bytes ({len(pcm)/2/16000:.1f}s)")

    t0 = time.monotonic()
    chunk_size = 3200  # 100ms at 16kHz s16le mono
    silence_chunk = b"\x00" * chunk_size

    for i in range(0, len(pcm), chunk_size):
        chunk = pcm[i : i + chunk_size]
        await session.send_realtime_input(
            audio=types.Blob(mime_type="audio/pcm;rate=16000", data=chunk),
        )
        await asyncio.sleep(0.1)

    # 2s silence for VAD end-of-speech detection
    for _ in range(20):
        await session.send_realtime_input(
            audio=types.Blob(mime_type="audio/pcm;rate=16000", data=silence_chunk),
        )
        await asyncio.sleep(0.1)

    events = await collect_events(session, label, t0)
    return {"label": label, "input_text": text, "lang": lang, "pcm_bytes": len(pcm), "events": events}

## 실행: EN + KO 각 1 turn

In [ ]:
api_key = os.environ.get("GEMINI_API_KEY")
assert api_key, "Set GEMINI_API_KEY environment variable"

client = genai.Client(api_key=api_key)
config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    realtime_input_config=types.RealtimeInputConfig(
        automatic_activity_detection=types.AutomaticActivityDetection(
            disabled=False,
            start_of_speech_sensitivity=types.StartSensitivity.START_SENSITIVITY_HIGH,
            end_of_speech_sensitivity=types.EndSensitivity.END_SENSITIVITY_HIGH,
        ),
    ),
)

results = []

async with client.aio.live.connect(model=MODEL, config=config) as session:
    print(f"Connected to {MODEL}")

    r1 = await run_audio_turn(session, "Hello, what is two plus two?", lang="en", label="audio-en")
    results.append(r1)
    await asyncio.sleep(2)

    r2 = await run_audio_turn(session, "내일 오후 3시 미팅 잡아줘", lang="ko", label="audio-ko")
    results.append(r2)

print("Done")

## 결과: transcript vs modelTurn 도착 타이밍 표

In [ ]:
# Build timing summary table
print(f"{'Label':<12} {'Input':<35} {'Transcript(ms)':>15} {'1st Model(ms)':>15} {'Delta(ms)':>10} {'Verdict'}")
print("-" * 100)

for r in results:
    transcript_ms = None
    first_model_ms = None
    transcript_text = ""

    for ev in r["events"]:
        if ev["type"] == "inputTranscription" and transcript_ms is None:
            transcript_ms = ev["elapsed_ms"]
            transcript_text = ev.get("text", "")
        if ev["type"] == "modelTurn" and first_model_ms is None:
            first_model_ms = ev["elapsed_ms"]

    if transcript_ms is not None and first_model_ms is not None:
        delta = transcript_ms - first_model_ms
        verdict = "BEFORE" if delta <= 0 else "AFTER"
        print(f"{r['label']:<12} {r['input_text']:<35} {transcript_ms:>15.1f} {first_model_ms:>15.1f} {delta:>+10.1f} {verdict}")
    else:
        print(f"{r['label']:<12} {r['input_text']:<35} {'N/A':>15} {'N/A':>15} {'N/A':>10} MISSING")

## 결론

| 테스트 | inputTranscription | 첫 modelTurn | Delta | 판정 |
|---|---|---|---|---|
| EN: "Hello, what is 2+2?" | +4282.8ms | +4284.6ms | **-1.8ms** | transcript **먼저** |
| KO: "내일 오후 세시 미팅 잡아줘" | +4279.8ms | +4281.0ms | **-1.2ms** | transcript **먼저** |

**`inputTranscription`이 `modelTurn`보다 항상 ~1-2ms 먼저 도착.**

### Design decision

- **Parallel pipeline 채택 확정** (Gihwang's design)
- Audio는 Gemini에 그대로 forward → transcript 도착 즉시 classifier 시작
- modelTurn 응답은 Response Buffer (~100ms) 에 지연 → classifier 결과 따라 flush / drop
- `gemini-3.1-flash-live-preview`는 **AUDIO 전용** (TEXT modality → Internal Error)